---

# Concept Explanation

## What is Fine-Tuning?

> Taking a pretrained model and training it on your specific task

---

## Example

A pretrained **BERT** already knows:

* Language
* Grammar
* Context

You teach it:

* Sentiment analysis
* Classification
* Your custom dataset

---

## Key Idea

```text
Pretrained Knowledge + Your Data = Powerful Model
```

---

# Why It Matters

Without fine-tuning:

* Model is general 

With fine-tuning:

* Model becomes **task-specific** 

---

# Simple Example

Think of BERT like:

- A graduate student in language

Fine-tuning =
- Teaching them a **specific job**

---


# Import Libraries

In [1]:
!pip install transformers datasets

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

import re
import numpy as np
import pandas as pd
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.metrics import accuracy_score

# Create Sample Dataset

In [2]:
df = pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    words = text.split()
    
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    return " ".join(words)

In [4]:
df["preprocessed_dataset"] = df["review"].apply(preprocess_text)

In [5]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

In [6]:
df["preprocessed_dataset"]

0        one reviewer mentioned watching oz episode hoo...
1        wonderful little production filming technique ...
2        thought wonderful way spend time hot summer we...
3        basically family little boy jake think zombie ...
4        petter mattei love time money visually stunnin...
                               ...                        
49995    thought movie right good job creative original...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    catholic taught parochial elementary school nu...
49998    going disagree previous comment side maltin on...
49999    one expects star trek movie high art fan expec...
Name: preprocessed_dataset, Length: 50000, dtype: object

In [7]:
dataset = Dataset.from_pandas(df)

# Load Tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Tokenization Function

In [9]:
def tokenize(example):
    return tokenizer(
        example["preprocessed_dataset"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [10]:
dataset = dataset.remove_columns([
    "review",                 
    "preprocessed_dataset",   
    "sentiment"                
])

In [11]:
dataset = dataset.train_test_split(test_size=0.1)

# Load Model

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Training Arguments

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,
    num_train_epochs=3,               
    learning_rate=2e-5,              
)

# Accuracy Metric

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# Trainer

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
500,0.751397
1000,0.590988
1500,0.526995
2000,0.403861
2500,0.379705
3000,0.342422
3500,0.258383
4000,0.239019


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4221, training_loss=0.42557636704723917, metrics={'train_runtime': 1904.6223, 'train_samples_per_second': 70.88, 'train_steps_per_second': 2.216, 'total_flos': 8879998118400000.0, 'train_loss': 0.42557636704723917, 'epoch': 3.0})

In [17]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6698479652404785,
 'eval_accuracy': 0.8952,
 'eval_runtime': 24.8238,
 'eval_samples_per_second': 201.419,
 'eval_steps_per_second': 6.325,
 'epoch': 3.0}

# Test Prediction

In [23]:
device = model.device 

test_texts = [
    "This movie was amazing!",
    "I did not like this film at all.",
    "An absolute masterpiece!",
    "It was boring and too long."
]

inputs = tokenizer(
    test_texts,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

preds = torch.argmax(outputs.logits, dim=1)

for text, pred in zip(test_texts, preds):
    label = "positive" if pred.item() == 1 else "negative"
    print(f"Text: {text}\nPrediction: {label}\n")

Text: This movie was amazing!
Prediction: positive

Text: I did not like this film at all.
Prediction: negative

Text: An absolute masterpiece!
Prediction: positive

Text: It was boring and too long.
Prediction: negative



---

# Limitations

- Needs GPU for speed
- Requires tuning (learning rate, epochs)
- Can overfit small datasets

---

# Mini Summary

* Fine-tuning = adapt pretrained model
* Uses small dataset efficiently
* Very powerful in real-world NLP
* Done using Hugging Face Trainer

---

# Connection to Previous Notebook

* Before: used pretrained models
* Now: trained them on your data

---